# 第 6 周 · 周四：SQL 思维实战——经典分析题与面试真题详解

> 难度：⭐⭐⭐⭐ | 定位：通过8道经典分析题训练"业务问题→SQL"的翻译能力，附带面试评分标准和常见追问
> 前置依赖：W5-W6全部SQL知识 | 后续用途：面试实战、日常分析工作、技术方案设计
> 面试地位：大厂SQL面试90%的题目都可以归入这几类经典模式，掌握它们=通过SQL面

---

## 📘 区块1：核心概念与第一性原理

### 1.1 分析题的"翻译"思维——面试和工作的核心能力

面试和工作中遇到的SQL分析题，本质上都是在考察你能否把"业务语言"翻译成"SQL语言"。这个翻译过程分三步：

**第一步：识别分析模式**
这个业务问题属于哪种模式？TopN？留存？漏斗？累计？分层？面试题通常就是这几种模式的排列组合。

**第二步：确定数据变换步骤**
数据需要经过哪几步变换才能得到答案？每一步用什么SQL工具？在纸上画出数据流。

**第三步：选择SQL语法并处理边界**
每步对应的具体SQL语法是什么？NULL怎么处理？重复数据怎么处理？大数据量下性能如何？

### 1.2 六大经典分析模式总览与面试权重

| 模式 | 典型问题 | 核心SQL工具 | 面试频率 | 难度 |
|------|---------|-----------|--------|------|
| 排名TopN | 每个品类销售额前10的商品 | ROW_NUMBER + PARTITION BY | 极高(90%+) | ⭐⭐ |
| 留存分析 | 新增用户次日/7日/30日留存 | LEFT JOIN + DATEDIFF | 极高(80%+) | ⭐⭐⭐ |
| 连续行为 | 连续登录≥7天的用户 | ROW_NUMBER + 日期减法 | 高(60%+) | ⭐⭐⭐⭐ |
| 累计指标 | GMV累计到当前日期的总和 | SUM OVER + ORDER BY | 高(70%+) | ⭐⭐ |
| 漏斗分析 | 首页→搜索→加购→支付的转化 | COUNT DISTINCT + CASE WHEN | 高(60%+) | ⭐⭐⭐ |
| 占比分析 | 各品类GMV占比 | SUM(SUM) OVER + GROUP BY | 中(50%+) | ⭐ |

### 1.3 面试官的评分标准(知己知彼)

面试官在评估你的SQL答案时，通常按以下权重打分：
1. **正确性(40%)**：逻辑是否正确，边界条件是否处理了NULL、0值、重复数据。这是及格线——逻辑错了，其他再好也没用。
2. **思路清晰(30%)**：能否解释清楚为什么这样写，步骤是否合理。很多面试官更看重思路——因为语法可以查，思路不行。
3. **可读性(20%)**：是否用了CTE命名(而非嵌套子查询)，格式是否规范，有没有注释。这反映了你的工程素养。
4. **性能意识(10%)**：是否考虑了大数据场景下的性能优化，是否提到了分区裁剪、数据倾斜等。这是"会用SQL"和"精通SQL"的分水岭。

### 1.4 面试中的常见追问及应对策略

面试官写出基础SQL后，通常会追问：

**追问1**："如果这个表有10亿行，你的SQL还能跑吗？"
应对：讨论分区裁剪、先过滤再JOIN、用窗口函数替代自连接、避免COUNT(DISTINCT)改用GROUP BY。

**追问2**："如果某个品类只有3个用户，你的TopN会怎样？"
应对：ROW_NUMBER正常返回3行，不会报错。但要主动提出"需要关注数据分布"的业务意识。

**追问3**："如果当天没有新增用户，留存率怎么显示？"
应对：当天新增为0时，留存率无意义，可以显示为NULL或过滤掉。要区分"0%留存"(有新增但无人留存)和"无数据"(没有新增)。

**追问4**："能不能不用窗口函数实现同样的功能？"
应对：可以用自连接+GROUP BY(但代码复杂且性能差)。提到窗口函数是MySQL 8.0+才有的，老版本需要用变量或自连接。

## 🧩 区块2：经典分析题详解(8道必会题)

### 题目1：新用户次日留存率

**问题**：给定用户登录表logins(user_id, login_date)，计算每日新增用户的次日留存率。

**分析思路**：
1. 找出每个用户的首次登录日期(first_dt)
2. 用LEFT JOIN查出首次登录+1天仍然登录的用户
3. 按日期分组，计算COUNT(DISTINCT次日用户) / COUNT(DISTINCT首日用户)

**关键细节**：
- 必须用LEFT JOIN而非INNER JOIN：保证分母完整(没有留存的用户也要计入)
- 同一用户同一天可能多次登录：需要DISTINCT或先GROUP BY去重
- 新增用户太少(<30人)的日期，留存率波动很大，不要过度解读
- 7日留存只需改一行：DATE_ADD(first_date, INTERVAL 7 DAY)

**SQL实现**：
```sql
WITH first_dt AS (
  SELECT user_id, MIN(login_date) AS first_date
  FROM logins
  GROUP BY user_id
)
SELECT
  f.first_date,
  COUNT(DISTINCT f.user_id) AS new_users,
  COUNT(DISTINCT l.user_id) AS retained_users,
  ROUND(COUNT(DISTINCT l.user_id) * 100.0
    / COUNT(DISTINCT f.user_id), 1) AS retention_pct
FROM first_dt f
LEFT JOIN logins l
  ON f.user_id = l.user_id
  AND l.login_date = DATE_ADD(f.first_date, INTERVAL 1 DAY)
GROUP BY f.first_date
ORDER BY f.first_date;
```

### 题目2：连续登录≥N天的用户(Gap-and-Island经典)

**问题**：找出连续登录≥5天的用户及起止日期。

**核心洞察**：连续的日期减去各自的序号后，会得到相同的参考日期。这就是gap-and-island的精髓——将"连续"转化为"相同"，然后GROUP BY分组。

**四步法**：
```sql
WITH t1 AS (
  SELECT DISTINCT user_id, login_date AS dt FROM logins  -- Step1: 去重
),
t2 AS (
  SELECT user_id, dt,
    ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY dt) AS rn  -- Step2: 编号
  FROM t1
),
t3 AS (
  SELECT user_id, dt,
    DATE_SUB(dt, INTERVAL rn DAY) AS grp  -- Step3: 日期减编号→分组键
  FROM t2
),
t4 AS (
  SELECT user_id, grp,
    MIN(dt) AS start_date,
    MAX(dt) AS end_date,
    COUNT(*) AS consecutive_days  -- Step4: 分组统计
  FROM t3
  GROUP BY user_id, grp
)
SELECT * FROM t4
WHERE consecutive_days >= 5
ORDER BY consecutive_days DESC;
```

### 题目3：累计到当前行的总和

**问题**：计算每个用户按时间排序的累计消费额。

**SQL**：
```sql
SELECT user_id, order_date, amount,
  SUM(amount) OVER (PARTITION BY user_id ORDER BY order_date
    ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS cum_amount
FROM orders
ORDER BY user_id, order_date;
```

**变体**：7天滚动和 = ROWS BETWEEN 6 PRECEDING AND CURRENT ROW；移动平均 = AVG替代SUM。

### 题目4：每个品类GMV及占比

**问题**：计算每个品类的GMV及其占总GMV的百分比。

**SQL**：
```sql
SELECT category,
  ROUND(SUM(amount), 0) AS gmv,
  ROUND(SUM(amount) * 100.0 / SUM(SUM(amount)) OVER(), 1) AS pct
FROM orders
GROUP BY category
ORDER BY gmv DESC;
```
**技巧**：SUM(SUM(amount)) OVER() = 窗口函数嵌套聚合。内层SUM按GROUP BY聚合每个品类，外层SUM OVER计算所有品类的总计。

### 题目5：漏斗分析——转化率

**问题**：计算"首页浏览→商品详情→加入购物车→下单支付"的转化漏斗。

**SQL**：
```sql
WITH funnel AS (
  SELECT
    COUNT(DISTINCT CASE WHEN event='page_view' THEN user_id END) AS step1,
    COUNT(DISTINCT CASE WHEN event='product_detail' THEN user_id END) AS step2,
    COUNT(DISTINCT CASE WHEN event='add_cart' THEN user_id END) AS step3,
    COUNT(DISTINCT CASE WHEN event='purchase' THEN user_id END) AS step4
  FROM user_events
)
SELECT step1 AS page_view, step2 AS detail, step3 AS add_cart, step4 AS purchase,
  ROUND(step2*100.0/step1,1) AS pv_to_detail,
  ROUND(step3*100.0/step2,1) AS detail_to_cart,
  ROUND(step4*100.0/step3,1) AS cart_to_purchase,
  ROUND(step4*100.0/step1,1) AS overall_conversion
FROM funnel;
```

### 题目6：复购率

**SQL**：
```sql
WITH user_month AS (
  SELECT user_id,
    DATE_FORMAT(order_date, '%Y-%m') AS month, COUNT(*) AS order_count
  FROM orders GROUP BY user_id, DATE_FORMAT(order_date, '%Y-%m')
)
SELECT month, COUNT(*) AS total_users,
  SUM(CASE WHEN order_count >= 2 THEN 1 ELSE 0 END) AS repurchase_users,
  ROUND(SUM(CASE WHEN order_count >= 2 THEN 1 ELSE 0 END)*100.0/COUNT(*),1) AS rate
FROM user_month GROUP BY month ORDER BY month;
```

### 题目7：各品类前10%用户的GMV贡献(二八法则验证)

**SQL**：
```sql
WITH user_cat AS (
  SELECT user_id, category, SUM(amount) AS total
  FROM orders GROUP BY user_id, category
),
ranked AS (
  SELECT *, PERCENT_RANK() OVER (PARTITION BY category ORDER BY total DESC) AS pct_rk
  FROM user_cat
),
cat_total AS (
  SELECT category, SUM(total) AS total_gmv FROM user_cat GROUP BY category
)
SELECT r.category,
  ROUND(SUM(r.total),0) AS top10_gmv,
  ROUND(c.total_gmv,0) AS total_gmv,
  ROUND(SUM(r.total)/c.total_gmv*100,1) AS pct
FROM ranked r JOIN cat_total c ON r.category=c.category
WHERE r.pct_rk <= 0.1
GROUP BY r.category ORDER BY pct DESC;
```
**典型结果**：前10%用户通常贡献40-60%的GMV。如果某个品类这个比例特别高(>70%)，说明该品类严重依赖极少数大客户——业务风险高。

### 题目8：新用户首单品类分析

**SQL**：
```sql
WITH first_order AS (
  SELECT user_id, MIN(order_date) AS first_date FROM orders GROUP BY user_id
),
first_category AS (
  SELECT o.user_id, o.category FROM orders o
  JOIN first_order f ON o.user_id=f.user_id AND o.order_date=f.first_date
)
SELECT category, COUNT(*) AS first_order_count,
  ROUND(COUNT(*)*100.0/SUM(COUNT(*)) OVER(),1) AS pct
FROM first_category GROUP BY category ORDER BY first_order_count DESC;
```

### 8道题的 SQL 速查卡

| # | 题型 | 核心SQL模式 | 关键细节 |
|---|------|-----------|--------|
| 1 | 留存 | LEFT JOIN + DATE_ADD + 1 DAY | 必须LEFT JOIN(保留无留存的用户) |
| 2 | 连续 | ROW_NUMBER + DATE_SUB(rn) + GROUP BY grp | 日期减序号=gap-and-island精髓 |
| 3 | 累计 | SUM OVER + ORDER BY + UNBOUNDED PRECEDING | ROWS帧控制范围 |
| 4 | 占比 | GROUP BY + SUM(SUM) OVER() | 聚合嵌套窗口函数 |
| 5 | 漏斗 | COUNT DISTINCT + CASE WHEN event | 关注转化率最低的环节 |
| 6 | 复购 | GROUP BY user + COUNT + CASE WHEN >=2 | 定义要明确：≥2单 |
| 7 | 分层 | PERCENT_RANK + WHERE <= 0.1 | 二八法则验证 |
| 8 | 首单 | MIN(date) + JOIN回原表 | 新用户首单行为分析 |

## 💻 区块3：动手练习——8道题实战

生成综合数据集：登录表(200用户x15天) + 订单表(400行x4品类) + 事件表(用于漏斗分析)

In [ ]:
import sqlite3, pandas as pd, numpy as np
np.random.seed(42); conn=sqlite3.connect(':memory:')

logins=[]
for uid in range(1,201):
    s=np.random.choice(pd.date_range('2026-01-01','2026-01-15'))
    for d in sorted(np.random.choice(pd.date_range(s,'2026-01-31'),
        size=np.random.randint(3,20),replace=False)):
        logins.append((uid,d.strftime('%Y-%m-%d')))
pd.DataFrame(logins,columns=['uid','dt']).to_sql('logins',conn,index=False)

orders=pd.DataFrame({
    'uid':np.random.choice(range(1,201),400),
    'cat':np.random.choice(['A','B','C','D'],400),
    'amt':np.round(np.random.lognormal(4,0.7,400),2),
    'dt':np.random.choice(pd.date_range('2026-01-01','2026-01-31'),400)
})
orders['dt']=orders['dt'].astype(str)
orders.to_sql('orders',conn,index=False)

print("题1：每日新用户次日留存率")
q1="""WITH first_dt AS (SELECT uid,MIN(dt) AS fd FROM logins GROUP BY uid)
SELECT fd,COUNT(DISTINCT f.uid) AS new_u,
  COUNT(DISTINCT l.uid) AS ret,
  ROUND(COUNT(DISTINCT l.uid)*100.0/COUNT(DISTINCT f.uid),1) AS rate
FROM first_dt f LEFT JOIN logins l ON f.uid=l.uid AND l.dt=DATE(f.fd,'+1 day')
GROUP BY fd HAVING new_u>=5 ORDER BY fd DESC LIMIT 5"""
print(pd.read_sql(q1,conn))

print("\n题2：连续登录>=5天的用户及起止日期")
q2="""WITH t1 AS (SELECT DISTINCT uid,dt FROM logins),
t2 AS (SELECT *,ROW_NUMBER() OVER(PARTITION BY uid ORDER BY dt) rn FROM t1),
t3 AS (SELECT *,DATE(dt,'-'||rn||' days') grp FROM t2),
t4 AS (SELECT uid,grp,MIN(dt) start_dt,MAX(dt) end_dt,COUNT(*) days FROM t3 GROUP BY uid,grp)
SELECT * FROM t4 WHERE days>=5 ORDER BY days DESC LIMIT 8"""
print(pd.read_sql(q2,conn))

print("\n题3：每个用户的累计消费")
q3="""SELECT uid,dt,amt,
  SUM(amt) OVER(PARTITION BY uid ORDER BY dt) AS cumsum
FROM orders ORDER BY uid,dt LIMIT 12"""
print(pd.read_sql(q3,conn))

print("\n题4：每个品类GMV及占比")
q4="""SELECT cat,
  ROUND(SUM(amt),0) AS gmv,
  ROUND(SUM(amt)*100.0/SUM(SUM(amt)) OVER(),1) AS pct
FROM orders GROUP BY cat ORDER BY gmv DESC"""
print(pd.read_sql(q4,conn))


### 步骤2：进阶练习——漏斗分析与复购率

In [ ]:
# 题5：漏斗分析
events=pd.DataFrame({
    'user_id':np.random.choice(range(1,101),300),
    'event':np.random.choice(['page_view','product_detail','add_cart','purchase'],300,
        p=[0.40,0.30,0.20,0.10])
})
events.to_sql('events',conn,index=False)

q5="""WITH funnel AS (
  SELECT
    COUNT(DISTINCT CASE WHEN event='page_view' THEN user_id END) AS step1,
    COUNT(DISTINCT CASE WHEN event='product_detail' THEN user_id END) AS step2,
    COUNT(DISTINCT CASE WHEN event='add_cart' THEN user_id END) AS step3,
    COUNT(DISTINCT CASE WHEN event='purchase' THEN user_id END) AS step4
  FROM events
)
SELECT step1 AS page_view, step2 AS detail, step3 AS add_cart, step4 AS purchase,
  ROUND(step2*100.0/step1,1) AS pv2detail,
  ROUND(step3*100.0/step2,1) AS detail2cart,
  ROUND(step4*100.0/step3,1) AS cart2purchase,
  ROUND(step4*100.0/step1,1) AS overall_rate
FROM funnel"""
print("漏斗分析：")
print(pd.read_sql(q5,conn))

# 题6：复购率
print("\n复购率分析：")
q6="""WITH uo AS (SELECT uid, COUNT(*) AS cnt FROM orders GROUP BY uid)
SELECT COUNT(*) AS total, SUM(CASE WHEN cnt>=2 THEN 1 ELSE 0 END) AS rpt,
  ROUND(SUM(CASE WHEN cnt>=2 THEN 1 ELSE 0 END)*100.0/COUNT(*),1) AS rate
FROM uo"""
print(pd.read_sql(q6,conn))

# 题7：二八法则验证
print("\n题7：各品类前20%用户GMV占比：")
q7="""WITH user_cat AS (
  SELECT uid, cat, SUM(amt) AS total FROM orders GROUP BY uid, cat
),
ranked AS (
  SELECT *, PERCENT_RANK() OVER(PARTITION BY cat ORDER BY total DESC) AS pct_rk FROM user_cat
),
cat_total AS (SELECT cat, SUM(total) AS total_gmv FROM user_cat GROUP BY cat)
SELECT r.cat, ROUND(SUM(r.total),0) AS top20_gmv,
  ROUND(c.total_gmv,0) AS total_gmv,
  ROUND(SUM(r.total)/c.total_gmv*100,1) AS pct
FROM ranked r JOIN cat_total c ON r.cat=c.cat
WHERE r.pct_rk<=0.2 GROUP BY r.cat ORDER BY pct DESC"""
print(pd.read_sql(q7,conn))


### 步骤3：挑战练习——面试综合题

"找出过去30天中，消费金额排名前20%且购买>=2个品类的用户，统计他们的品类偏好分布和平均客单价。"

这道题综合了：时间过滤 + 排名分层 + 条件筛选 + 品类偏好 + 聚合统计——典型的大厂面试综合题。

In [ ]:
# 综合面试题
print("面试综合题：高价值多品类用户的偏好分析")
q_boss = """WITH user_stats AS (
  SELECT uid, SUM(amt) AS total_spend,
    COUNT(DISTINCT cat) AS cat_count,
    COUNT(*) AS order_count, ROUND(AVG(amt),1) AS avg_order
  FROM orders GROUP BY uid
),
ranked AS (
  SELECT *, PERCENT_RANK() OVER(ORDER BY total_spend DESC) AS pct_rk FROM user_stats
),
top_users AS (
  SELECT uid, total_spend, cat_count, avg_order
  FROM ranked WHERE pct_rk <= 0.2 AND cat_count >= 2
),
user_fav AS (
  SELECT uid, cat,
    ROW_NUMBER() OVER(PARTITION BY uid ORDER BY SUM(amt) DESC) AS rn
  FROM orders WHERE uid IN (SELECT uid FROM top_users)
  GROUP BY uid, cat
)
SELECT uf.cat AS fav_category,
  COUNT(DISTINCT uf.uid) AS user_count,
  ROUND(AVG(tu.total_spend),0) AS avg_spend,
  ROUND(AVG(tu.avg_order),1) AS avg_order_value
FROM user_fav uf JOIN top_users tu ON uf.uid=tu.uid
WHERE uf.rn=1
GROUP BY uf.cat ORDER BY user_count DESC"""
print(pd.read_sql(q_boss, conn))
print("\n这道题综合了5种SQL模式：过滤+排名+筛选+分组偏好+聚合统计")
print("面试时，主动向面试官解释每一步的用途和选择——这比写出正确SQL更重要")


## 📊 区块4：面试实战场景模拟

### 场景1：面试官出题——考察分析思维

面试官："给你一个订单表，怎么找出每个品类消费金额排名前10的用户？"

**你应该这样回答(展示思维过程)**：
"这个问题属于'分组TopN'模式。我的思路是：首先按用户和品类聚合得到每人每个品类的消费总额；然后用ROW_NUMBER按品类分组、按消费降序编号；最后筛选编号≤10。这里有几个细节需要确认——同一品类如果有并列第10名，用ROW_NUMBER只取一个(严格10个)，还是用RANK允许并列？这取决于业务需求。另外，如果品类用户总数很少(<10人)，Top10就全部返回，不会报错。"

**面试官心理**：这个候选人不仅会写SQL，还会考虑边界情况和业务需求——给分！

### 场景2：面试官追问边界情况

面试官："如果一个品类只有5个用户怎么办？"

**你的回答**："ROW_NUMBER只会返回5行(rn=1到5)，WHERE rn<=10仍然有效，不会报错。但这种情况提醒我们——如果很多品类的用户都很少，Top10就没意义了。分析之前应该先看一下品类用户数的分布，确认Top10分析对该数据集是否有意义。"

### 场景3：面试官追问性能

面试官："订单表有10亿行，这个查询能跑得动吗？"

**你的回答**："需要关注几点：第一，窗口函数的PARTITION BY会不会造成数据倾斜——如果某个品类占了80%的数据，大部分计算都集中在一个节点；第二，先按品类和用户聚合(用GROUP BY)可以减少进入窗口函数的数据量；第三，如果只需要最近的数据，先用WHERE缩小时间范围。在Hive/Spark环境下，可能还需要调整shuffle分区数。"

### 场景4：面试官问"不用窗口函数怎么做"

面试官："如果数据库不支持窗口函数(比如MySQL 5.7)，你怎么做？"

**你的回答**："可以用自连接+GROUP BY实现。比如TopN：用自连接LEFT JOIN，条件是同一个品类内amount比自己高的行数，再GROUP BY统计这个行数<N就是TopN。但这种方式在大数据量下性能很差(O(n²))——这也是为什么窗口函数是SQL标准中非常重要的特性。如果有大量这类需求，建议升级数据库版本或使用支持窗口函数的查询引擎。"

## ✅ 区块5：分级检验清单

### 🟢 初级检验(必须能不看笔记独立完成的8道题)
- [ ] 排名TopN：ROW_NUMBER + PARTITION BY + WHERE rn <= N
- [ ] 次日留存：LEFT JOIN + DATE_ADD + INTERVAL 1 DAY
- [ ] 连续登录：ROW_NUMBER + DATE_SUB(rn) + GROUP BY grp(四步法)
- [ ] 累计指标：SUM OVER + ORDER BY + UNBOUNDED PRECEDING
- [ ] GMV占比：GROUP BY + SUM(SUM) OVER()
- [ ] 漏斗分析：COUNT DISTINCT + CASE WHEN event
- [ ] 复购率：GROUP BY user + HAVING COUNT >= 2
- [ ] 二八法则：PERCENT_RANK + WHERE <= 0.1(或0.2)

### 🟡 中级检验(能解释原理和应对追问)
- [ ] 能解释为什么留存必须用LEFT JOIN(而非INNER JOIN)
- [ ] 能解释gap-and-island中"日期减序号"的数学原理
- [ ] 能处理面试追问：数据倾斜、NULL处理、边界情况
- [ ] 能说出SUM(SUM) OVER()的执行逻辑(聚合嵌套窗口)
- [ ] 能对比ROW_NUMBER/RANK/DENSE_RANK的适用场景

### 🔴 高级检验(能举一反三和主动展示深度)
- [ ] 能解决组合型面试题(排名+分层+对比的多模式组合)
- [ ] 能不用窗口函数实现TopN和累计值(展示底层理解)
- [ ] 能讨论不同数据库(MySQL/PG/Hive/Spark)的优化策略差异
- [ ] 能主动思考分析结果的业务含义和可能偏差
- [ ] 能设计面试题的变体和扩展(展现分析深度)

**高级面试的关键**：不仅写出SQL，还能讨论："这个分析告诉我们什么？"、"数据量扩大1000倍怎么办？"、"有哪些潜在偏差需要警惕？"——这三个追问是区分中级和高级分析师的关键。

## 🔗 区块6：跨模块连接

| 连接方向 | 关联内容 | 为什么有关 |
|---------|---------|----------|
| **← 前置依赖** | W5 所有SQL工具 | 窗口函数/JOIN/CTE是解题的基础 |
| **← 前置依赖** | W6D1-D3 实战方法 | 复杂SQL拆解/画像/活动评估是真实场景 |
| **→ 直接应用** | 大厂SQL面试 | 本篇覆盖了90%的面试题型 |
| **→ 后续应用** | W10综合案例 | 综合案例需要组合多种SQL模式 |
| **业务连接** | 数据分析日常工作 | TopN/留存/漏斗/复购是日常分析核心 |

### 面试必背SQL模式卡片

```
1. TopN:     ROW_NUMBER() OVER(PARTITION BY col ORDER BY val DESC) rn ... WHERE rn <= N
2. 留存:     LEFT JOIN ON a.id=b.id AND b.date = DATE_ADD(a.date, INTERVAL N DAY)
3. 连续:     ROW_NUMBER ... DATE_SUB(date, INTERVAL rn DAY) grp ... GROUP BY grp
4. 累计:     SUM(col) OVER(PARTITION BY id ORDER BY date) cum
5. 占比:     SUM(col)*100.0 / SUM(SUM(col)) OVER() pct
6. 漏斗:     COUNT(DISTINCT CASE WHEN event='X' THEN user END)
7. 分层:     PERCENT_RANK() OVER(ORDER BY val DESC) ... WHERE pct <= 0.1
8. 复购:     GROUP BY user ... HAVING COUNT(*) >= 2
```

## 📝 区块7：当日小结

> **SQL面试的核心是"业务问题→SQL模式"的翻译能力。8道经典题覆盖了90%的面试题型，必须能不看笔记独立写出。面试官最看重的不是语法正确(40%)，而是思路清晰(30%)+可读性(20%)+性能意识(10%)。能主动讨论边界情况(NULL/倾斜/性能)是高级面试的加分项——这是"会用SQL"和"精通SQL"的分水岭。**

**三条金句带回家**：
1. 面试遇到SQL题，先说出分析思路再写代码——思路正确比语法正确更能赢得面试官
2. 8道经典题是你的"SQL九九乘法表"——要能不看笔记默写，就像乘法口诀一样脱口而出
3. 主动讨论边界情况(NULL、数据倾斜、性能)是区分中级和高级的分水岭

### 🔖 课后拓展

1. **练习**：找3道大厂SQL面试真题(牛客网/LeetCode)，不看答案独立完成，每道控制在10分钟内

2. **思考**：如果不能用窗口函数(比如限制只能用标准SQL-92)，8道题中哪些能实现？哪些不能？怎么改写？

3. **面试准备**：准备一个30秒开场——"我熟练使用SQL进行数据分析，最擅长用窗口函数和CTE解决复杂分析问题，比如留存分析、用户分层和漏斗分析"

4. **预习**：明天的SQL性能调优——将让你在面试中展现出"不只是能写SQL，还能写好SQL"的能力

---

*03analyst_thinking · Week 6 Day 4 · 2026-05-22*